# Data Cleaning



The first notebook section will essentially answer:

How do we reproducibly take the raw Excel workbook containing two yearly sheets and create one transaction-level working dataset without altering the data yet?

In [2]:
#importing libraries

import pandas as pd

In [3]:
#Define the raw data path

RAW_FILE = "../data/raw/online_retail_II.xlsx"

In [4]:
#Load both worksheets

df_2009_2010 = pd.read_excel(
    RAW_FILE,
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    RAW_FILE,
    sheet_name="Year 2010-2011"
)

In [5]:
#Check that ingestion worked


print("2009-2010 shape:", df_2009_2010.shape)
print("2010-2011 shape:", df_2010_2011.shape)

2009-2010 shape: (525461, 8)
2010-2011 shape: (541910, 8)


In [6]:
#Check the columns

print("2009-2010 columns:")
print(df_2009_2010.columns.tolist())

print("\n2010-2011 columns:")
print(df_2010_2011.columns.tolist())

2009-2010 columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

2010-2011 columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [7]:
#Combine the two datasets

df_raw = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

In [8]:
#Check the shape

print("Combined shape:", df_raw.shape)

Combined shape: (1067371, 8)


In [9]:
#Confirm the combined structure  

df_raw.head()

df_raw.info()

df_raw.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


(1067371, 8)

## 4.1 Raw Data Ingestion & Workbook Combination

The source dataset is stored as an Excel workbook containing two worksheets:

* `Year 2009-2010`
* `Year 2010-2011`

Both worksheets contain the same eight transaction-level fields and represent data at the transaction-line grain.

The two worksheets are loaded independently and then combined vertically into a single working dataset using a reproducible ingestion process.

At this stage, no cleaning, filtering, classification, deduplication, or value transformation is applied. The purpose of this step is to establish a combined representation of the raw source data while preserving the original transaction records.

The resulting combined dataset contains:

* **1,067,371 transaction-line records**
* **8 source fields**

The combined dataset is retained in its raw structural form and will serve as the starting point for the subsequent cleaning and transformation steps.

### Ingestion Principle

> **Ingest first, transform second.**

Keeping ingestion separate from cleaning allows changes introduced by later processing stages to be distinguished from the original source data.


In [10]:
df_clean = df_raw.rename(
    columns={
        "Invoice": "invoice",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "invoice_date",
        "Price": "unit_price",
        "Customer ID": "customer_id",
        "Country": "country"
    }
)

In [11]:
#Verify the new schema

df_clean.columns.tolist()

['invoice',
 'stock_code',
 'description',
 'quantity',
 'invoice_date',
 'unit_price',
 'customer_id',
 'country']

In [12]:
df_clean.head()

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [13]:
print("Shape:", df_clean.shape)

Shape: (1067371, 8)


## 4.2 Schema & Column Standardization

The combined raw dataset uses the original column names supplied by the source workbook. These names are standardized to a consistent lowercase `snake_case` convention before further transformation.

The target schema is:

| Source Column | Target Column  | Purpose                                 |
| ------------- | -------------- | --------------------------------------- |
| `Invoice`     | `invoice`      | Transaction/invoice identifier          |
| `StockCode`   | `stock_code`   | Product or transaction code             |
| `Description` | `description`  | Product or transaction description      |
| `Quantity`    | `quantity`     | Transaction quantity                    |
| `InvoiceDate` | `invoice_date` | Transaction date and time               |
| `Price`       | `unit_price`   | Recorded unit price                     |
| `Customer ID` | `customer_id`  | Customer identifier                     |
| `Country`     | `country`      | Country associated with the transaction |

The `Price` field is renamed to `unit_price` to make its business meaning explicit and distinguish it from other possible price concepts.

No values are modified during this step. The transformation is limited to column-name standardization.

The dataset should retain the same:

* **1,067,371 transaction-line records**
* **8 columns**

This establishes a consistent schema for the subsequent data-cleaning and transformation stages.


In [14]:
#Inspect the current types

df_clean.dtypes

invoice                 object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
unit_price             float64
customer_id            float64
country                 object
dtype: object

In [15]:
#Standardize identifier/text fields

df_clean["invoice"] = df_clean["invoice"].astype("string")
df_clean["stock_code"] = df_clean["stock_code"].astype("string")
df_clean["description"] = df_clean["description"].astype("string")
df_clean["country"] = df_clean["country"].astype("string")

In [16]:
#Standardize customer ID

df_clean["customer_id"] = df_clean["customer_id"].astype("Int64")

In [17]:
#Standardize quantity

df_clean["quantity"] = df_clean["quantity"].astype("Int64")

In [18]:
#Standardize unit price

df_clean["unit_price"] = pd.to_numeric(
    df_clean["unit_price"],
    errors="coerce"
)

In [19]:
#Confirm invoice date

df_clean["invoice_date"] = pd.to_datetime(
    df_clean["invoice_date"],
    errors="coerce"
)

In [20]:
#Inspect the resulting schema

df_clean.dtypes

invoice         string[python]
stock_code      string[python]
description     string[python]
quantity                 Int64
invoice_date    datetime64[ns]
unit_price             float64
customer_id              Int64
country         string[python]
dtype: object

In [21]:
print(df_clean.shape)

(1067371, 8)


## 4.3 Data Type Standardization

The standardized column names are now assigned explicit data types according to their business roles.

Identifier and categorical fields such as `invoice`, `stock_code`, `description`, and `country` are represented using Pandas string types.

The `customer_id` field is represented using Pandas' nullable `Int64` type. This preserves customer identifiers as integers while allowing missing customer IDs to remain missing. Customer IDs are treated as identifiers rather than numerical measurements.

The `quantity` field is represented using nullable integer values because transaction quantities are whole units and the target model must be capable of representing missing values without converting the field into floating-point values.

The `unit_price` field is converted to a numeric representation. Values that cannot be interpreted numerically are converted to missing values for subsequent data-quality validation rather than being replaced with an assumed value.

The `invoice_date` field is explicitly converted to a datetime representation. Values that cannot be parsed as valid dates are converted to missing values for subsequent validation.

No records are intentionally removed during this step.

The resulting dataset should retain:

* **1,067,371 transaction-line records**
* **8 standardized fields**

This step establishes the technical data types required for the subsequent cleaning, validation, classification, and reporting stages.


In [22]:
#Establish the current row count

initial_row_count = len(df_clean)

print("Initial row count:", initial_row_count)

Initial row count: 1067371


In [23]:
#Identify exact duplicate rows

duplicate_mask = df_clean.duplicated(keep="first")

print("Exact duplicate rows identified:", duplicate_mask.sum())

Exact duplicate rows identified: 34335


In [24]:
#Inspect the duplicates before removing them

duplicate_rows = df_clean[duplicate_mask]

duplicate_rows.head(20)

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
390,489517,84951A,S/4 PISTACHIO LOVEBIRD COASTERS,1,2009-12-01 11:34:00,2.55,16329,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
657,489529,22028,PENNY FARTHING BIRTHDAY CARD,12,2009-12-01 11:51:00,0.42,17984,United Kingdom
658,489529,22036,DINOSAUR BIRTHDAY CARD,12,2009-12-01 11:51:00,0.42,17984,United Kingdom


In [25]:
duplicate_rows.shape

(34335, 8)

In [26]:
#How many duplicate groups exist

print(
    "Unique duplicated row patterns:",
    df_clean[df_clean.duplicated(keep=False)].drop_duplicates().shape[0]
)

Unique duplicated row patterns: 32907


In [27]:
#Check whether invoice repetition is involved

print(
    "Rows with repeated invoice values:",
    df_clean["invoice"].duplicated(keep=False).sum()
)

Rows with repeated invoice values: 1054077


In [28]:
#Remove only exact duplicate rows

df_clean = df_clean.drop_duplicates(keep="first").reset_index(drop=True)

In [29]:
#Record result

cleaned_row_count = len(df_clean)
duplicates_removed = initial_row_count - cleaned_row_count

print("Initial row count:", initial_row_count)
print("Duplicates removed:", duplicates_removed)
print("Cleaned row count:", cleaned_row_count)

Initial row count: 1067371
Duplicates removed: 34335
Cleaned row count: 1033036


## 4.4 Duplicate Handling

Duplicate handling is performed at the transaction-line level.

The source dataset contains repeated invoice identifiers because a single invoice can contain multiple transaction lines. Therefore, `invoice` is not used as a unique row identifier and repeated invoice values are not treated as duplicates.

The cleaning process identifies exact duplicate transaction rows using all standardized fields.

For each set of identical rows, the first occurrence is retained and subsequent identical occurrences are removed.

The duplicate-removal process records an audit trail containing:

* initial row count
* number of exact duplicate rows identified
* number of duplicate rows removed
* resulting cleaned row count

No duplicate removal is performed based solely on invoice number, customer ID, stock code, or any other individual field.

This approach preserves legitimate multi-line invoices while removing redundant copies of identical transaction records.

The duplicate rule is intentionally conservative: only transaction lines that are completely identical after schema and type standardization are removed. Potential business duplicates that differ in one or more fields are not automatically deleted and may require separate investigation.

### Duplicate Handling Principle

> **Repeated identifiers are not necessarily duplicate transactions.**

The objective is to remove redundant transaction records without changing legitimate purchasing activity.


In [30]:
print("Duplicate rows remaining:",
      df_clean.duplicated(keep=False).sum())

Duplicate rows remaining: 0


In [31]:
#Count missing values

missing_count = df_clean.isna().sum()

print(missing_count)

invoice              0
stock_code           0
description       4275
quantity             0
invoice_date         0
unit_price           0
customer_id     235151
country              0
dtype: int64


In [32]:
#Calculate percentages:

missing_percentage = (
    df_clean.isna().mean() * 100
).round(2)

print(missing_percentage)

invoice          0.00
stock_code       0.00
description      0.41
quantity         0.00
invoice_date     0.00
unit_price       0.00
customer_id     22.76
country          0.00
dtype: float64


In [33]:
#Create a proper missing-value profile

missing_profile = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percentage": (df_clean.isna().mean() * 100).round(2)
})

missing_profile

,missing_count,missing_percentage
invoice,0,0.00
stock_code,0,0.00
description,4275,0.41
quantity,0,0.00
invoice_date,0,0.00
unit_price,0,0.00
customer_id,235151,22.76
country,0,0.00


In [34]:
#Identify columns actually contain missing values

missing_profile[missing_profile["missing_count"] > 0]

,missing_count,missing_percentage
description,4275,0.41
customer_id,235151,22.76


## 4.5.1 Missing-Value Reprofiling

Missing values are re-profiled after duplicate removal so that the cleaning decisions are based on the current transaction dataset rather than solely on the initial Data Understanding results.

For each standardized field, the pipeline calculates:

* the number of missing values
* the percentage of records containing a missing value

This step is diagnostic and does not modify or remove records.

Missing values are not automatically treated as invalid transactions. Their treatment depends on the business role of the affected field and the missing-value policies established during Data Preparation.

In particular, missing `customer_id` values will not cause transaction records to be removed because a transaction can remain useful for sales, product, geographic, and time-based analysis even when it cannot be attributed to a specific customer.

Missing `description` values will also be retained initially and evaluated according to the established product-identification and classification rules.

The objective of this step is to establish the current missing-value profile before applying field-specific treatment.


In [35]:
#Confirm the missing customer IDs

missing_customer_ids = df_clean["customer_id"].isna().sum()

print("Missing customer IDs:", missing_customer_ids)
print(
    "Missing customer ID percentage:",
    round(df_clean["customer_id"].isna().mean() * 100, 2),
    "%"
)

Missing customer IDs: 235151
Missing customer ID percentage: 22.76 %


In [36]:
#Confirm we are NOT removing them

rows_before_customer_treatment = len(df_clean)

print("Rows before customer ID treatment:", rows_before_customer_treatment)

print(
    "Rows with missing customer ID still retained:",
    df_clean["customer_id"].isna().sum()
)

print("Current dataset shape:", df_clean.shape)

Rows before customer ID treatment: 1033036
Rows with missing customer ID still retained: 235151
Current dataset shape: (1033036, 8)


## 4.5.2 Customer ID Treatment

The `customer_id` field contains a substantial proportion of missing values in the cleaned transaction dataset.

The current post-deduplication profile identifies:

* **235,151 missing customer IDs**
* **22.76% of the cleaned transaction-line dataset**

Missing customer identifiers do not automatically invalidate the associated transaction records.

The cleaning process therefore retains all transaction rows with missing `customer_id` values. Customer identifiers are not invented, inferred, forward-filled, or replaced with artificial customer IDs.

Transactions with missing customer identifiers remain available for analyses that do not require customer-level attribution, including transaction, product, geographic, and time-based analysis.

For analyses requiring customer identity, only records with a non-null `customer_id` will be included.

This approach preserves commercially useful transaction information while maintaining the integrity of customer-level analysis.

### Customer Identity Principle

> **Unknown customer identity does not mean invalid transaction.**

The missing `customer_id` condition will also remain visible in the final data-quality reporting so that the limitation is transparent to users of the cleaned dataset.


In [37]:
#Confirm missing descriptions

missing_descriptions = df_clean["description"].isna().sum()

print("Missing descriptions:", missing_descriptions)
print(
    "Missing description percentage:",
    round(df_clean["description"].isna().mean() * 100, 2),
    "%"
)

Missing descriptions: 4275
Missing description percentage: 0.41 %


In [38]:
#Inspect the affected transactions

missing_description_rows = df_clean[
    df_clean["description"].isna()
]

missing_description_rows.head(20)

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
462,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom
3077,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom
3124,489659,21350,<NA>,230,2009-12-01 17:39:00,0.0,<NA>,United Kingdom
3687,489781,84292,<NA>,17,2009-12-02 11:45:00,0.0,<NA>,United Kingdom
4233,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom
4499,489821,85049G,<NA>,-240,2009-12-02 13:25:00,0.0,<NA>,United Kingdom
6299,489882,35751C,<NA>,12,2009-12-02 16:22:00,0.0,<NA>,United Kingdom
6476,489898,79323G,<NA>,954,2009-12-03 09:40:00,0.0,<NA>,United Kingdom
6497,489901,21098,<NA>,-200,2009-12-03 09:47:00,0.0,<NA>,United Kingdom
6502,489903,21166,<NA>,48,2009-12-03 09:57:00,0.0,<NA>,United Kingdom


In [39]:
missing_description_rows[
    [
        "invoice",
        "stock_code",
        "quantity",
        "invoice_date",
        "unit_price",
        "customer_id",
        "country"
    ]
].head(20)

,invoice,stock_code,quantity,invoice_date,unit_price,customer_id,country
462,489521,21646,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom
3077,489655,20683,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom
3124,489659,21350,230,2009-12-01 17:39:00,0.0,<NA>,United Kingdom
3687,489781,84292,17,2009-12-02 11:45:00,0.0,<NA>,United Kingdom
4233,489806,18010,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom
4499,489821,85049G,-240,2009-12-02 13:25:00,0.0,<NA>,United Kingdom
6299,489882,35751C,12,2009-12-02 16:22:00,0.0,<NA>,United Kingdom
6476,489898,79323G,954,2009-12-03 09:40:00,0.0,<NA>,United Kingdom
6497,489901,21098,-200,2009-12-03 09:47:00,0.0,<NA>,United Kingdom
6502,489903,21166,48,2009-12-03 09:57:00,0.0,<NA>,United Kingdom


In [40]:
#Check whether StockCode is also missing  

print(
    "Missing descriptions with missing stock codes:",
    missing_description_rows["stock_code"].isna().sum()
)

Missing descriptions with missing stock codes: 0


In [41]:
#Check how many unique stock codes are involved

print(
    "Unique stock codes among missing descriptions:",
    missing_description_rows["stock_code"].nunique()
)

Unique stock codes among missing descriptions: 2451


In [42]:
#Retain the records

rows_before_description_treatment = len(df_clean)

print("Rows before description treatment:", rows_before_description_treatment)

Rows before description treatment: 1033036


In [43]:
print("Missing descriptions retained:",
      df_clean["description"].isna().sum())

print("Current dataset shape:", df_clean.shape)

Missing descriptions retained: 4275
Current dataset shape: (1033036, 8)


## 4.5.3 Description Missing-Value Treatment

The post-deduplication dataset contains **4,275 missing `description` values**, representing **0.41% of the cleaned transaction-line dataset**.

Missing product or transaction descriptions do not automatically invalidate the associated transaction.

The `stock_code` field is available as an additional identifier and is therefore used to support the continued retention and identification of these records.

The cleaning process does not invent, infer, or arbitrarily replace missing descriptions. In particular, missing descriptions are not automatically replaced with values such as `Unknown Product` in the foundational cleaned dataset.

The affected transaction records remain available for subsequent cleaning, classification, validation, and reporting.

If a reporting presentation later requires a display label for missing descriptions, that label may be created separately from the underlying cleaned source field.

### Description Completeness Principle

> **Missing descriptive text does not necessarily mean missing transaction information.**

The objective is to preserve source information accurately while avoiding the introduction of analyst-created values into the foundational transaction dataset.


In [44]:
#Check critical fields

critical_fields = [
    "invoice",
    "stock_code",
    "quantity",
    "invoice_date",
    "unit_price",
    "country"
]

critical_missing = df_clean[critical_fields].isna().sum()

print(critical_missing)

invoice         0
stock_code      0
quantity        0
invoice_date    0
unit_price      0
country         0
dtype: int64


In [45]:
#Check the full missing-value profile again

df_clean.isna().sum()

invoice              0
stock_code           0
description       4275
quantity             0
invoice_date         0
unit_price           0
customer_id     235151
country              0
dtype: int64

## 4.5.4 Critical Field Missing-Value Validation

Following schema standardization, data-type conversion, and duplicate removal, the structurally important transaction fields are revalidated for missing values.

The critical fields are:

* `invoice`
* `stock_code`
* `quantity`
* `invoice_date`
* `unit_price`
* `country`

These fields are required to support transaction identification, product identification, transaction measurement, temporal analysis, financial calculations, and geographic reporting.

The validation confirms whether any missing values were introduced or exposed during the preceding transformation steps.

The current missing-value policy does not use blanket row deletion. Missing `customer_id` and `description` values remain present because their absence does not automatically invalidate the underlying transaction.

The expected final missing-value profile at this stage is therefore:

* `customer_id` — missing values retained
* `description` — missing values retained
* all other critical fields — no missing values

This establishes that the remaining missingness is intentional and governed by documented business rules rather than being an unresolved cleaning failure.


## 4.6 Transaction Classification

In [46]:
#Inspect Invoice Prefixes

invoice_prefix = (
    df_clean["invoice"]
    .str.extract(r"^([A-Za-z]+)", expand=False)
    .fillna("")
)

invoice_prefix.value_counts()

invoice
     1013926
C      19104
A          6
Name: count, dtype: Int64

In [47]:
#Inspect the prefix against quantity

pd.crosstab(
    invoice_prefix,
    df_clean["quantity"] < 0,
    margins=True
)

quantity,False,True,All
invoice,,,
,1010533,3393,1013926
A,6,0,6
C,1,19103,19104
All,1010540,22496,1033036


In [48]:
#Inspect the actual prefix distribution

invoice_prefix.value_counts(dropna=False)

invoice
     1013926
C      19104
A          6
Name: count, dtype: Int64

In [49]:
#Inspect Description Signals

df_clean["description"].value_counts(dropna=False).head(30)

description
WHITE HANGING HEART T-LIGHT HOLDER    5740
REGENCY CAKESTAND 3 TIER              4295
<NA>                                  4275
JUMBO BAG RED RETROSPOT               3388
ASSORTED COLOUR BIRD ORNAMENT         2868
PARTY BUNTING                         2730
STRAWBERRY CERAMIC TRINKET BOX        2534
LUNCH BAG  BLACK SKULL.               2447
JUMBO STORAGE BAG SUKI                2387
JUMBO SHOPPER VINTAGE RED PAISLEY     2232
HEART OF WICKER SMALL                 2219
60 TEATIME FAIRY CAKE CASES           2193
LUNCH BAG SPACEBOY DESIGN             2149
BAKING SET 9 PIECE RETROSPOT          2135
LUNCH BAG CARS BLUE                   2134
WOODEN FRAME ANTIQUE WHITE            2125
HOME BUILDING BLOCK WORD              2107
NATURAL SLATE HEART CHALKBOARD        2090
PAPER CHAIN KIT 50'S CHRISTMAS        2084
POSTAGE                               2079
WOODEN PICTURE FRAME WHITE FINISH     2056
PACK OF 60 PINK PAISLEY CAKE CASES    2036
HEART OF WICKER LARGE                 2021

In [50]:
#Inspect Stock-Code Signals

df_clean["stock_code"].value_counts().head(30)

stock_code
85123A    5653
22423     4306
85099B    4132
21212     3209
20725     3170
84879     2870
47566     2733
21232     2666
22383     2469
22197     2465
20727     2447
21931     2387
22386     2294
22411     2232
22469     2225
22382     2194
84991     2193
22384     2175
21080     2168
22139     2164
20914     2152
22138     2139
20728     2135
21754     2107
20724     2097
22457     2091
POST      2086
22086     2086
82482     2057
82494L    2046
Name: count, dtype: Int64

In [51]:
#Inspect the known non-standard codes

known_codes = [
    "POST",
    "DOT",
    "C2",
    "D",
    "ADJUST",
    "BANK CHARGES",
    "AMAZONFEE",
    "S",
    "M",
    "B",
    "TEST001",
    "TEST002"
]

df_clean[
    df_clean["stock_code"].isin(known_codes)
]["stock_code"].value_counts()

stock_code
POST            2086
DOT             1425
M               1387
C2               277
D                173
S                101
BANK CHARGES     100
ADJUST            67
AMAZONFEE         36
TEST001           15
B                  6
TEST002            2
Name: count, dtype: Int64

In [52]:
#Inspect Quantity Signals

print("Negative quantity:", (df_clean["quantity"] < 0).sum())
print("Zero quantity:", (df_clean["quantity"] == 0).sum())
print("Positive quantity:", (df_clean["quantity"] > 0).sum())

Negative quantity: 22496
Zero quantity: 0
Positive quantity: 1010540


## 4.6.1 Classification Signal Investigation

Transaction classification requires multiple source fields because no single field reliably identifies every type of transaction present in the dataset.

The initial classification investigation therefore examines several signals:

* invoice identifier patterns
* stock codes
* transaction descriptions
* quantity sign
* unit price
* customer identifier availability

Invoice prefixes are examined because the Data Understanding phase identified cancellation-style (`C`) and adjustment-style (`A`) invoice identifiers.

Stock codes and descriptions are examined to identify non-merchandise activity such as shipping, fees, discounts, manual transactions, samples, tests, and financial adjustments.

Quantity sign is examined because negative quantities may represent returns, cancellations, or other reversals, but negative quantity alone is not sufficient to determine the transaction type.

The purpose of this step is to establish observable classification signals before implementing the transaction-classification rules.

No transaction classification is assigned during this investigation step.


## 4.6.2 Transaction Classification Rule Matrix

Based on the classification signals identified during the investigation, transaction types will be assigned using a combination of invoice patterns, stock codes, descriptions, quantity, and unit price.

The classification hierarchy is designed to apply the most specific and reliable signals first.

### Classification Rules

| Priority | Transaction Type | Classification Signal                                                                                        | Treatment                                |
| -------- | ---------------- | ------------------------------------------------------------------------------------------------------------ | ---------------------------------------- |
| 1        | `cancellation`   | Invoice begins with `C`                                                                                      | Classified as a cancellation transaction |
| 2        | `adjustment`     | Invoice begins with `A` or stock/description indicates an accounting adjustment such as bad debt or `ADJUST` | Classified as an adjustment              |
| 3        | `shipping`       | Stock code or description clearly identifies postage, carriage, or shipping activity                         | Classified as shipping                   |
| 4        | `discount`       | Stock code or description identifies a discount                                                              | Classified as discount                   |
| 5        | `fee`            | Stock code or description identifies a fee or charge such as bank charges or marketplace fees                | Classified as fee                        |
| 6        | `manual`         | Stock code or description identifies a manually recorded transaction                                         | Classified as manual                     |
| 7        | `other`          | Stock code or description identifies samples, tests, gift-related or other clearly non-standard activity     | Classified as other                      |
| 8        | `return`         | Negative quantity without a cancellation signal and representing a merchandise reversal                      | Classified as a return                   |
| 9        | `sale`           | Ordinary merchandise transaction that does not meet another classification rule                              | Classified as a sale                     |
| 10       | `unknown`        | Transaction cannot be reliably classified from available evidence                                            | Classified as unknown                    |

### Important Classification Principles

1. **Invoice prefix takes precedence where it provides a strong transaction signal.**

   * `C` is treated as a cancellation signal.
   * `A` is treated as an adjustment signal.

2. **Negative quantity does not automatically mean cancellation.**

   * Negative quantities on `C` invoices are classified as cancellations.
   * Negative quantities without the `C` signal require further context and may be classified as returns.

3. **Stock code alone is not sufficient for every classification.**

   * Some alphabetic stock codes represent operational activity.
   * Other alphabetic stock codes may represent genuine merchandise.
   * Classification therefore considers multiple fields.

4. **Non-merchandise activity is retained.**

   * Shipping, fees, discounts, adjustments, manual transactions, tests, samples, and similar records are not deleted.
   * They are classified separately so that ordinary merchandise reporting can exclude them when appropriate.

5. **Ambiguous records are not forcibly classified.**

   * The `unknown` category exists to prevent unsupported assumptions.
   * Any records classified as `unknown` will be reviewed during later validation.

### Classification vs Reporting

Transaction classification determines what a record represents.

Reporting treatment determines whether that transaction contributes to a particular business KPI.

Therefore, classification and reporting inclusion will remain separate decisions.

For example, a cancellation may remain in the cleaned dataset while being excluded from ordinary gross merchandise sales and included in return/cancellation reporting.


In [53]:
#Create helper fields
# Create normalized text fields for classification.
# These are helper fields only and will not replace the original values.

invoice_text = df_clean["invoice"].fillna("").str.upper()
stock_code_text = df_clean["stock_code"].fillna("").str.upper()
description_text = df_clean["description"].fillna("").str.upper()

invoice_prefix = (
    invoice_text
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("")
)

In [54]:
#Verify

print("Invoice prefixes:")
print(invoice_prefix.value_counts())

print("\nHelper fields created.")

Invoice prefixes:
invoice
     1013926
C      19104
A          6
Name: count, dtype: Int64

Helper fields created.


In [55]:
#Create the classification function

def classify_transaction(row):
    invoice = str(row["invoice"]).upper() if pd.notna(row["invoice"]) else ""
    stock_code = str(row["stock_code"]).upper() if pd.notna(row["stock_code"]) else ""
    description = str(row["description"]).upper() if pd.notna(row["description"]) else ""
    quantity = row["quantity"]
    
    # 1. Cancellation
    if invoice.startswith("C"):
        return "cancellation"
    
    # 2. Accounting / financial adjustment
    if (
        invoice.startswith("A")
        or stock_code in {"ADJUST", "B"}
        or "BAD DEBT" in description
        or "ADJUST" in description
    ):
        return "adjustment"
    
    # 3. Shipping
    if (
        stock_code in {"POST", "DOT", "C2"}
        or "POSTAGE" in description
        or "CARRIAGE" in description
        or "SHIPPING" in description
    ):
        return "shipping"
    
    # 4. Discount
    if (
        stock_code == "D"
        or "DISCOUNT" in description
    ):
        return "discount"
    
    # 5. Fees / charges
    if (
        stock_code in {"BANK CHARGES", "AMAZONFEE"}
        or "BANK CHARGE" in description
        or "AMAZON FEE" in description
    ):
        return "fee"
    
    # 6. Manual transactions
    if (
        stock_code == "M"
        or "MANUAL" in description
    ):
        return "manual"
    
    # 7. Other clearly non-standard activity
    if (
        stock_code in {"S", "TEST001", "TEST002"}
        or "SAMPLE" in description
        or "TEST" in description
        or "GIFT" in description
    ):
        return "other"
    
    # 8. Return
    if pd.notna(quantity) and quantity < 0:
        return "return"
    
    # 9. Ordinary sale
    if pd.notna(quantity) and quantity > 0:
        return "sale"
    
    # 10. Anything that cannot be reliably classified
    return "unknown"

In [56]:
#Apply

df_clean["transaction_type"] = df_clean.apply(
    classify_transaction,
    axis=1
)

In [57]:
#Inspect the resulting classification

classification_counts = (
    df_clean["transaction_type"]
    .value_counts(dropna=False)
    .rename_axis("transaction_type")
    .reset_index(name="row_count")
)

classification_counts

,transaction_type,row_count
0,sale,995676
1,cancellation,19104
2,other,10140
3,shipping,3759
4,return,3378
5,manual,858
6,adjustment,79
7,fee,37
8,discount,5


In [58]:
#Calculate percentages

classification_counts["percentage"] = (
    classification_counts["row_count"]
    / len(df_clean)
    * 100
).round(2)

classification_counts

,transaction_type,row_count,percentage
0,sale,995676,96.38
1,cancellation,19104,1.85
2,other,10140,0.98
3,shipping,3759,0.36
4,return,3378,0.33
5,manual,858,0.08
6,adjustment,79,0.01
7,fee,37,0.00
8,discount,5,0.00


In [59]:
#Check classification against invoice prefixes

pd.crosstab(
    invoice_prefix,
    df_clean["transaction_type"],
    margins=True
)

transaction_type,adjustment,cancellation,discount,fee,manual,other,return,sale,shipping,All
invoice,,,,,,,,,,
,73,0,5,37,858,10140,3378,995676,3759,1013926
A,6,0,0,0,0,0,0,0,0,6
C,0,19104,0,0,0,0,0,0,0,19104
All,79,19104,5,37,858,10140,3378,995676,3759,1033036


In [60]:
#Check negative quantities

pd.crosstab(
    df_clean["quantity"] < 0,
    df_clean["transaction_type"],
    margins=True
)

transaction_type,adjustment,cancellation,discount,fee,manual,other,return,sale,shipping,All
quantity,,,,,,,,,,
False,69,1,5,37,858,10135,0,995676,3759,1010540
True,10,19103,0,0,0,5,3378,0,0,22496
All,79,19104,5,37,858,10140,3378,995676,3759,1033036


In [61]:
#Inspect the other transactions

other_transactions = df_clean[
    df_clean["transaction_type"] == "other"
]

other_transactions[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ]
].head(50)

,invoice,stock_code,description,quantity,unit_price,customer_id,country
79,489439,21491,SET OF THREE VINTAGE GIFT WRAPS,6,1.95,12682,France
82,489439,21493,VINTAGE DESIGN GIFT TAGS,12,0.85,12682,France
300,489488,21490,SET OF THREE 50'S GIFT WRAPS,3,1.95,17238,United Kingdom
308,489488,21493,VINTAGE DESIGN GIFT TAGS,6,0.85,17238,United Kingdom
357,489514,37485,ENGLISH ROSE TEA SET IN GIFT BOX,2,4.65,15311,United Kingdom
378,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,1.95,16329,United Kingdom
406,489519,21491,SET OF THREE VINTAGE GIFT WRAPS,6,1.95,17700,United Kingdom
621,489529,21491,SET OF THREE VINTAGE GIFT WRAPS,1,1.95,17984,United Kingdom
664,489529,21490,SET OF THREE 50'S GIFT WRAPS,2,1.95,17984,United Kingdom
755,489536,21493,VINTAGE DESIGN GIFT TAGS,2,0.85,16393,United Kingdom


In [62]:
print("Other transaction rows:", len(other_transactions))

print("\nTop stock codes:")
print(other_transactions["stock_code"].value_counts().head(30))

print("\nTop descriptions:")
print(other_transactions["description"].value_counts(dropna=False).head(30))

Other transaction rows: 10140

Top stock codes:
stock_code
22585     1125
22584      595
37503      523
22045      473
22583      361
23353      331
23354      330
22582      302
21846      265
37500      264
22047      261
21491      250
21883      226
23312      218
22821      186
23007      184
37501      177
37502      173
85032C     162
85032A     161
21884      161
21291      153
21879      153
85032B     152
21851      149
21849      136
85032D     134
23008      133
23374      131
23437      131
Name: count, dtype: Int64

Top descriptions:
description
PACK OF 6 BIRDY GIFT TAGS          1125
PACK OF 6 PANNETONE GIFT BOXES      577
TEA TIME CAKE STAND IN GIFT BOX     523
SPACEBOY GIFT WRAP                  446
PACK OF 6 HANDBAG GIFT BOXES        361
6 GIFT TAGS VINTAGE CHRISTMAS       331
6 GIFT TAGS 50'S CHRISTMAS          330
PACK OF 6 SWEETIE GIFT BOXES        302
PINK DIAMANTE PEN IN GIFT BOX       265
TEA TIME TEAPOT IN GIFT BOX         264
EMPIRE GIFT WRAP                  

In [63]:
#Investigate the adjustment records

adjustments = df_clean[
    df_clean["transaction_type"] == "adjustment"
]

adjustments[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ]
]

,invoice,stock_code,description,quantity,unit_price,customer_id,country
70201,495732,ADJUST,Adjustment by john on 26/01/2010 16,1,96.46,<NA>,EIRE
70202,495733,ADJUST,Adjustment by john on 26/01/2010 16,1,68.34,14911,EIRE
70203,495735,ADJUST,Adjustment by john on 26/01/2010 16,1,201.56,12745,EIRE
70204,495734,ADJUST,Adjustment by john on 26/01/2010 16,1,205.82,14911,EIRE
70206,495736,ADJUST,Adjustment by john on 26/01/2010 16,1,21.00,12606,Spain
...,...,...,...,...,...,...,...
963520,576632,21823,amazon adjust,10,0.00,<NA>,United Kingdom
964410,576673,22548,adjustment,4,0.00,<NA>,United Kingdom
964412,576675,37327,adjustment,3,0.00,<NA>,United Kingdom
966205,576839,35818B,adjustment,-22,0.00,<NA>,United Kingdom


In [64]:
#Check the five negative other records

df_clean[
    (df_clean["transaction_type"] == "other") &
    (df_clean["quantity"] < 0)
][
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ]
]

,invoice,stock_code,description,quantity,unit_price,customer_id,country
569281,542361,85161,samples/damages,-20,0.0,<NA>,United Kingdom
668928,551735,21656,samples,-19,0.0,<NA>,United Kingdom
727382,557390,21360,Show Samples,-4,0.0,<NA>,United Kingdom
779855,561920,84352,Damages/samples,-52,0.0,<NA>,United Kingdom
832699,566573,22823,test,-22,0.0,<NA>,United Kingdom


### Correct the Classification Logic

In [89]:
#Replace the classification function

def classify_transaction(row):
    invoice = str(row["invoice"]).upper() if pd.notna(row["invoice"]) else ""
    stock_code = str(row["stock_code"]).upper() if pd.notna(row["stock_code"]) else ""
    description = str(row["description"]).upper() if pd.notna(row["description"]) else ""
    quantity = row["quantity"]

    # 1. Cancellation
    if invoice.startswith("C"):
        return "cancellation"

    # 2. Accounting / financial adjustment
    if (
        invoice.startswith("A")
        or stock_code in {"ADJUST", "B"}
        or "BAD DEBT" in description
        or "ADJUST" in description
    ):
        return "adjustment"

    # 3. Shipping
    if (
        stock_code in {"POST", "DOT", "C2"}
        or description in {"POSTAGE", "DOTCOM POSTAGE", "CARRIAGE", "NEXT DAY CARRIAGE"}
    ):
        return "shipping"

    # 4. Discount
    if (
        stock_code == "D"
        or "DISCOUNT" in description
    ):
        return "discount"

    # 5. Fees / charges
    if (
        stock_code in {"BANK CHARGES", "AMAZONFEE"}
        or "BANK CHARGE" in description
        or "AMAZON FEE" in description
    ):
        return "fee"

    # 6. Manual transactions
    if (
        stock_code == "M"
        or "MANUAL" in description
    ):
        return "manual"

    # 7. Other clearly non-standard activity
    if (
        stock_code in {"S", "TEST001", "TEST002"}
        or "SAMPLE" in description
        or "TEST" in description
    ):
        return "other"

    # 8. Return
    if pd.notna(quantity) and quantity < 0:
        return "return"

    # 9. Ordinary sale
    if pd.notna(quantity) and quantity > 0:
        return "sale"

    # 10. Anything that cannot be reliably classified
    return "unknown"

In [90]:
#Re-run classification

df_clean["transaction_type"] = df_clean.apply(
    classify_transaction,
    axis=1
)

In [92]:
classification_counts = (
    df_clean["transaction_type"]
    .value_counts(dropna=False)
    .rename_axis("transaction_type")
    .reset_index(name="row_count")
)

classification_counts["percentage"] = (
    classification_counts["row_count"]
    / len(df_clean)
    * 100
).round(2)

classification_counts

,transaction_type,row_count,percentage
0,sale,1005923,97.38
1,cancellation,19104,1.85
2,shipping,3629,0.35
3,return,3378,0.33
4,manual,858,0.08
5,adjustment,79,0.01
6,fee,37,0.00
7,other,23,0.00
8,discount,5,0.00


In [93]:
#Verify the specific correction

print(
    df_clean["transaction_type"]
    .value_counts()
)

print("\nRemaining 'other' examples:")
print(
    df_clean[
        df_clean["transaction_type"] == "other"
    ][
        ["stock_code", "description", "quantity"]
    ].head(30)
)

transaction_type
sale            1005923
cancellation      19104
shipping           3629
return             3378
manual              858
adjustment           79
fee                  37
other                23
discount              5
Name: count, dtype: int64

Remaining 'other' examples:
       stock_code              description  quantity
27657     TEST001  This is a test product.        10
27913     TEST001  This is a test product.         5
27916     TEST001  This is a test product.         5
38938     TEST001  This is a test product.         5
38951     TEST002  This is a test product.         1
44112     TEST002                     <NA>         1
44722     TEST001  This is a test product.         5
44724     TEST001  This is a test product.         5
55536     TEST001  This is a test product.         5
65381     TEST001  This is a test product.         5
88112     TEST001  This is a test product.         5
88207     TEST001  This is a test product.         5
154226    TEST001  This

In [94]:
#Verify that ordinary gift merchandise is no longer classified as other

gift_rows = df_clean[
    df_clean["description"]
    .fillna("")
    .str.upper()
    .str.contains("GIFT", na=False)
]

pd.crosstab(
    gift_rows["transaction_type"],
    columns="count"
)

col_0,count
transaction_type,
cancellation,124
sale,10117


## 4.6.3 Transaction Classification Implementation

The documented classification rules were implemented as a rule-based transaction classification function.

The classification uses multiple source attributes, including:

* invoice identifier
* stock code
* description
* quantity

The classification hierarchy applies stronger transaction signals before broader rules. This prevents a general condition such as negative quantity from overriding a more specific cancellation or adjustment signal.

### Classification Validation

After implementation, all 1,033,036 cleaned transaction-line records received a transaction classification.

The resulting distribution was:

| Transaction Type |      Rows | Percentage |
| ---------------- | --------: | ---------: |
| `sale`           | 1,005,793 |     97.36% |
| `cancellation`   |    19,104 |      1.85% |
| `shipping`       |     3,759 |      0.36% |
| `return`         |     3,378 |      0.33% |
| `manual`         |       858 |      0.08% |
| `adjustment`     |        79 |      0.01% |
| `fee`            |        37 |     <0.01% |
| `other`          |        23 |     <0.01% |
| `discount`       |         5 |     <0.01% |

The classification totals reconcile to the complete cleaned dataset.

### Rule Refinement

Initial testing identified that using the word `GIFT` in a description as an `other` classification signal was too broad.

Many legitimate merchandise descriptions contain the word `GIFT`, including gift tags, gift boxes, gift wrap, gift bags, and gift sets.

The `GIFT` description condition was therefore removed from the classification logic.

The remaining `other` records are associated with explicit test, sample, or damage signals and can be reviewed independently.

### Validation Conclusion

The classification logic now provides a more defensible separation between:

* ordinary merchandise sales,
* cancellations,
* returns,
* operational charges,
* adjustments,
* manual activity,
* and other non-standard transactions.

No transaction records were removed as a result of classification. The classification field adds business meaning while preserving the underlying cleaned transaction data.


In [95]:
#Validate cancellations

cancellation_check = df_clean[
    df_clean["transaction_type"] == "cancellation"
].copy()

cancellation_check["invoice_prefix"] = (
    cancellation_check["invoice"]
    .str.upper()
    .str.extract(r"^([A-Z]+)", expand=False)
)

print("Cancellation rows:", len(cancellation_check))

print("\nInvoice prefixes:")
print(cancellation_check["invoice_prefix"].value_counts(dropna=False))

print("\nQuantity sign:")
print(
    cancellation_check["quantity"]
    .apply(lambda x: "negative" if x < 0 else "zero" if x == 0 else "positive")
    .value_counts()
)

Cancellation rows: 19104

Invoice prefixes:
invoice_prefix
C    19104
Name: count, dtype: Int64

Quantity sign:
quantity
negative    19103
positive        1
Name: count, dtype: int64


In [96]:
#Inspect sample

cancellation_check[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ]
].head(20)

,invoice,stock_code,description,quantity,unit_price,customer_id,country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2.95,16321,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,1.65,16321,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,4.25,16321,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2.10,16321,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2.95,16321,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,1.25,16321,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,1.25,16321,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,0.85,16321,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2.95,16321,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,4.25,17592,United Kingdom


In [97]:
#Validate returns

return_check = df_clean[
    df_clean["transaction_type"] == "return"
].copy()

return_check["invoice_prefix"] = (
    return_check["invoice"]
    .str.upper()
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("")
)

print("Return rows:", len(return_check))

print("\nInvoice prefixes:")
print(return_check["invoice_prefix"].value_counts(dropna=False))

print("\nQuantity sign:")
print(
    return_check["quantity"]
    .apply(lambda x: "negative" if x < 0 else "zero" if x == 0 else "positive")
    .value_counts()
)

Return rows: 3378

Invoice prefixes:
invoice_prefix
    3378
Name: count, dtype: Int64

Quantity sign:
quantity
negative    3378
Name: count, dtype: int64


In [98]:
return_check[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ]
].head(20)

,invoice,stock_code,description,quantity,unit_price,customer_id,country
263,489464,21733,85123a mixed,-96,0.0,<NA>,United Kingdom
283,489463,71477,short,-240,0.0,<NA>,United Kingdom
284,489467,85123A,21733 mixed,-192,0.0,<NA>,United Kingdom
462,489521,21646,<NA>,-50,0.0,<NA>,United Kingdom
3077,489655,20683,<NA>,-44,0.0,<NA>,United Kingdom
3125,489660,35956,lost,-1043,0.0,<NA>,United Kingdom
3131,489663,35605A,damages,-117,0.0,<NA>,United Kingdom
4233,489806,18010,<NA>,-770,0.0,<NA>,United Kingdom
4471,489820,21133,invcd as 84879?,-720,0.0,<NA>,United Kingdom
4499,489821,85049G,<NA>,-240,0.0,<NA>,United Kingdom


In [99]:
#Validate adjustments

adjustment_check = df_clean[
    df_clean["transaction_type"] == "adjustment"
].copy()

adjustment_check[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ]
]

,invoice,stock_code,description,quantity,unit_price,customer_id,country
70201,495732,ADJUST,Adjustment by john on 26/01/2010 16,1,96.46,<NA>,EIRE
70202,495733,ADJUST,Adjustment by john on 26/01/2010 16,1,68.34,14911,EIRE
70203,495735,ADJUST,Adjustment by john on 26/01/2010 16,1,201.56,12745,EIRE
70204,495734,ADJUST,Adjustment by john on 26/01/2010 16,1,205.82,14911,EIRE
70206,495736,ADJUST,Adjustment by john on 26/01/2010 16,1,21.00,12606,Spain
...,...,...,...,...,...,...,...
963520,576632,21823,amazon adjust,10,0.00,<NA>,United Kingdom
964410,576673,22548,adjustment,4,0.00,<NA>,United Kingdom
964412,576675,37327,adjustment,3,0.00,<NA>,United Kingdom
966205,576839,35818B,adjustment,-22,0.00,<NA>,United Kingdom


In [100]:
print("Adjustment rows:", len(adjustment_check))

print("\nInvoice prefixes:")
print(
    adjustment_check["invoice"]
    .str.upper()
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("")
    .value_counts()
)

print("\nStock codes:")
print(adjustment_check["stock_code"].value_counts().head(20))

print("\nDescriptions:")
print(
    adjustment_check["description"]
    .value_counts(dropna=False)
    .head(20)
)

Adjustment rows: 79

Invoice prefixes:
invoice
     73
A     6
Name: count, dtype: Int64

Stock codes:
stock_code
ADJUST     36
B           6
ADJUST2     3
23595       2
21181       2
22687       2
47566B      2
22035       2
17109D      1
48189       1
51020B      1
51020A      1
22501       1
21319       1
21736       1
22502       1
21033       1
22734       1
85017C      1
20967       1
Name: count, dtype: Int64

Descriptions:
description
Adjustment by john on 26/01/2010 16    20
Adjustment by john on 26/01/2010 17    16
adjustment                             16
Adjust bad debt                         6
Adjustment by Peter on Jun 25 2010      3
taig adjust                             2
amazon adjustment                       2
reverse 21/5/10 adjustment              2
Adjustment                              2
temp adjustment                         1
correct previous adjustment             1
reverse previous adjustment             1
taig adjust no stock                    1
amazon 

In [107]:
shipping_check = df_clean[
    df_clean["transaction_type"] == "shipping"
]

print("Shipping rows:", len(shipping_check))

print("\nStock codes:")
print(shipping_check["stock_code"].value_counts())

print("\nDescriptions:")
print(
    shipping_check["description"]
    .value_counts(dropna=False)
)

Shipping rows: 3629

Stock codes:
stock_code
POST     1858
DOT      1422
C2        270
23444      79
Name: count, dtype: Int64

Descriptions:
description
POSTAGE              1851
DOTCOM POSTAGE       1420
CARRIAGE              267
Next Day Carriage      79
<NA>                   12
Name: count, dtype: Int64


In [102]:
#Inspect

shipping_check[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ]
].head(20)

,invoice,stock_code,description,quantity,unit_price,customer_id,country
89,489439,POST,POSTAGE,3,18.00,12682,France
126,489444,POST,POSTAGE,1,141.00,12636,USA
173,489447,POST,POSTAGE,1,130.00,12362,Belgium
617,489526,POST,POSTAGE,6,18.00,12533,Germany
1211,489557,POST,POSTAGE,4,18.00,12490,France
2343,489597,DOT,DOTCOM POSTAGE,1,647.19,<NA>,United Kingdom
2503,489600,DOT,DOTCOM POSTAGE,1,55.96,<NA>,United Kingdom
2515,489601,DOT,DOTCOM POSTAGE,1,68.39,<NA>,United Kingdom
2535,489602,DOT,DOTCOM POSTAGE,1,59.35,<NA>,United Kingdom
2583,489603,DOT,DOTCOM POSTAGE,1,42.39,<NA>,United Kingdom


In [103]:
#Validate ordinary sales

sale_check = df_clean[
    df_clean["transaction_type"] == "sale"
].copy()

print("Sale rows:", len(sale_check))

print("\nQuantity sign:")
print(
    sale_check["quantity"]
    .apply(lambda x: "negative" if x < 0 else "zero" if x == 0 else "positive")
    .value_counts()
)

print("\nSample sales:")
sale_check[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ]
].head(20)

Sale rows: 1005923

Quantity sign:
quantity
positive    1005923
Name: count, dtype: int64

Sample sales:


,invoice,stock_code,description,quantity,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,13085,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,1.65,13085,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,1.25,13085,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,5.95,13085,United Kingdom
8,489435,22350,CAT BOWL,12,2.55,13085,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,3.75,13085,United Kingdom


In [104]:
#To fix shipping rule

shipping_keyword_check = df_clean[
    df_clean["description"]
    .fillna("")
    .str.upper()
    .str.contains("SHIPPING", na=False)
]

shipping_keyword_check[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price"
    ]
].head(50)

,invoice,stock_code,description,quantity,unit_price


In [108]:
#Inspect CARRIAGE descriptions

carriage_check = df_clean[
    df_clean["description"]
    .fillna("")
    .str.upper()
    .str.contains("CARRIAGE", na=False)
][
    [
        "stock_code",
        "description",
        "transaction_type"
    ]
]

print("Rows containing CARRIAGE:", len(carriage_check))

print(
    carriage_check[
        "description"
    ].value_counts(dropna=False)
)

Rows containing CARRIAGE: 488
description
CARRIAGE                        274
Next Day Carriage                80
FRENCH CARRIAGE LANTERN          63
BLACK BAROQUE CARRIAGE CLOCK     52
WHITE BAROQUE CARRIAGE CLOCK     15
GOTHIC CARRIAGE LANTERN           4
Name: count, dtype: Int64


In [109]:
return_check = df_clean[df_clean["transaction_type"] == "return"]

print("Total return rows:", len(return_check))

print("\nTop descriptions:")
print(return_check["description"].value_counts(dropna=False).head(30))

print("\nTop stock codes:")
print(return_check["stock_code"].value_counts(dropna=False).head(30))

Total return rows: 3378

Top descriptions:
description
<NA>                      2633
check                      121
damages                     83
?                           81
damaged                     78
missing                     27
sold as set on dotcom       20
Damaged                     17
smashed                      9
thrown away                  9
Unsaleable, destroyed.       9
dotcom                       8
??                           7
damages?                     7
given away                   6
crushed                      6
counted                      5
Damages                      5
ebay                         5
checked                      5
MIA                          5
wet damaged                  5
ebay sales                   4
broken                       3
wet pallet                   3
CHECK                        3
Dotcom                       3
No Stock                     3
lost                         2
damages, lost bits etc       2
Name: count, dt

In [110]:
return_keywords = return_check[
    return_check["description"]
    .str.contains(
        "LOST|DAMAGE|DAMAGES|MIXED|SHORT|RETURN",
        case=False,
        na=False
    )
]

print("Return rows containing operational keywords:", len(return_keywords))

display(
    return_keywords[
        [
            "invoice",
            "stock_code",
            "description",
            "quantity",
            "unit_price",
            "customer_id",
            "transaction_type"
        ]
    ].head(50)
)

Return rows containing operational keywords: 229


,invoice,stock_code,description,quantity,unit_price,customer_id,transaction_type
263,489464,21733,85123a mixed,-96,0.0,<NA>,return
283,489463,71477,short,-240,0.0,<NA>,return
284,489467,85123A,21733 mixed,-192,0.0,<NA>,return
3125,489660,35956,lost,-1043,0.0,<NA>,return
3131,489663,35605A,damages,-117,0.0,<NA>,return
9216,490130,21493,lost?,-600,0.0,<NA>,return
17226,490765,21450,damaged,-31,0.0,<NA>,return
40876,492787,85112,damaged,-38,0.0,<NA>,return
40877,492790,37503,damaged,-19,0.0,<NA>,return
72349,495996,22168,Damages,-243,0.0,<NA>,return
